In [1]:
import glob

# from ravi.capabilities.knowledge.loaders.pdf_loader import PDFLoader

from pdfqa_rag.config import settings

DATA_DIR = settings.ROOT_DIR / "data" / "pdfQA-Benchmark" / "real-pdfQA"
pdf_paths = glob.glob(str(DATA_DIR / "01.2_Input_Files_PDF" / "FinanceBench" / "*.pdf"))

# loader = PDFLoader(extract_tables=True)
# docs = await loader.load(pdf_paths[0])

# print(f"Pages loaded : {len(docs)}")
# print(f"Sample text  : {docs[0].text[:200]}")

In [27]:
from datetime import datetime

pdf_date = "D:20180129174851-05'00'"

# Remove PDF prefix and fix timezone format
clean = pdf_date[2:].replace("'", "")

dt = datetime.strptime(clean, "%Y%m%d%H%M%S%z")

print(dt)

2018-01-29 17:48:51-05:00


In [28]:
import pypdf
from docling.chunking import HybridChunker
from docling.document_converter import DocumentConverter
from datetime import datetime

# 1. Extract PDF metadata via pypdf (one read, very fast)
def parse_pdf_date(date_str: str):
    if not date_str:
        return None

    try:
        # Example: D:20180129174851-05'00'
        clean = date_str.removeprefix("D:").replace("'", "")
        return datetime.strptime(clean, "%Y%m%d%H%M%S%z")
    except Exception:
        return None


def extract_pdf_metadata(path: str) -> dict:
    reader = pypdf.PdfReader(path)
    info = reader.metadata or {}

    return {
        "author": info.get("/Author"),
        "title": info.get("/Title"),
        "creator": info.get("/Creator"),
        "producer": info.get("/Producer"),
        "creation_date": parse_pdf_date(info.get("/CreationDate")),
        "mod_date": parse_pdf_date(info.get("/ModDate")),
        "keywords": info.get("/Keywords"),
        "subject": info.get("/Subject"),
       "pages": len(reader.pages),
    }

# 2. Parse + chunk with docling
converter = DocumentConverter()
chunker   = HybridChunker()

pdf_path = pdf_paths[0]
pdf_meta = extract_pdf_metadata(pdf_path)       # ← from pypdf
doc      = converter.convert(pdf_path).document  # ← from docling

# 3. Merge when building Document objects
for i, chunk in enumerate(chunker.chunk(doc)):
    text = chunker.contextualize(chunk)

    # pull page number from provenance
    page_no = None
    try:
        page_no = chunk.meta.doc_items[0].prov[0].page_no
    except (AttributeError, IndexError):
        pass

    metadata = {
        **pdf_meta,                          # author, title, dates from pypdf
        "headings": chunk.meta.headings or [],
        "chunk_index": i,
        "page_number": page_no,
        "filename": doc.origin.filename,
    }
    print(metadata)
    break


[INFO] 2026-06-11 20:06:42,353 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-11 20:06:42,360 [RapidOCR] download_file.py:60: File exists and is valid: /home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-11 20:06:42,361 [RapidOCR] main.py:57: Using /home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-11 20:06:42,445 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-11 20:06:42,448 [RapidOCR] download_file.py:60: File exists and is valid: /home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-11 20:06:42,448 [RapidOCR] main.py:57: Using /home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1180 > 512). Running this sequence through the model will result in indexing errors


{'author': 'EDGAR Online, a division of R.R. Donnelley & Sons Company', 'title': '0001065280-18-000069', 'creator': 'EDGAR PDF Generator', 'producer': 'EDGAR PDF Generator', 'creation_date': datetime.datetime(2018, 1, 29, 17, 48, 51, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400))), 'mod_date': datetime.datetime(2018, 1, 29, 17, 49, 2, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400))), 'keywords': '0001065280-18-000069; ; 10-K', 'subject': '10-K', 'pages': 73, 'headings': ['UNITED STATES SECURITIES AND EXCHANGE COMMISSION'], 'chunk_index': 0, 'page_number': 1, 'filename': 'NETFLIX_2017_10K.pdf'}


In [29]:
metadata

{'author': 'EDGAR Online, a division of R.R. Donnelley & Sons Company',
 'title': '0001065280-18-000069',
 'creator': 'EDGAR PDF Generator',
 'producer': 'EDGAR PDF Generator',
 'creation_date': datetime.datetime(2018, 1, 29, 17, 48, 51, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400))),
 'mod_date': datetime.datetime(2018, 1, 29, 17, 49, 2, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400))),
 'keywords': '0001065280-18-000069; ; 10-K',
 'subject': '10-K',
 'pages': 73,
 'headings': ['UNITED STATES SECURITIES AND EXCHANGE COMMISSION'],
 'chunk_index': 0,
 'page_number': 1,
 'filename': 'NETFLIX_2017_10K.pdf'}

In [4]:
file = open("converted_document.md", "w")
file.write(converted_docs.document.export_to_markdown())
file.close()

In [5]:
from docling.chunking import HybridChunker

chunker = HybridChunker()
chunks = chunker.chunk(converted_docs.document)
for chunk in chunks:
    context = chunker.contextualize(chunk)
    print(f"Chunk text : {context}")
    print("=" * 50)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1180 > 512). Running this sequence through the model will result in indexing errors


Chunk text : UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
_____________________________________________________________________
FORM 10-K
_____________________________________________________________________
(Mark One)
- [x] x ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2017
OR
- [ ] o TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to
Commission File Number: 001-35727
Chunk text : UNITED STATES SECURITIES AND EXCHANGE COMMISSION
_____________________________________________________________________
Netflix, Inc.
(Exact name of Registrant as specified in its charter)
_____________________________________________________________________
Delaware
77-0467272
(State or other jurisdiction of incorporation or organization)
(I.R.S. Employer Identification Number)
100 Winchester Circle Los Gatos, Californi

In [7]:
from pypdf import PdfReader

reader = PdfReader(pdf_paths[0])

reader.metadata

{'/Author': 'EDGAR Online, a division of R.R. Donnelley & Sons Company',
 '/Creator': 'EDGAR PDF Generator',
 '/Keywords': '0001065280-18-000069; ; 10-K',
 '/Producer': 'EDGAR PDF Generator',
 '/Subject': '10-K',
 '/Title': '0001065280-18-000069',
 '/CreationDate': "D:20180129174851-05'00'",
 '/ModDate': "D:20180129174902-05'00'"}